In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
from tqdm.notebook import tqdm

tfd = tfp.distributions
tfb = tfp.bijectors
from bakeoff.TensorFlow_Prob.run_tfp import make_conditioned_lp
from modulars.utils import load_config, load_best_values, exp_lognormal_moments
from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d
from modulars.tfp_rr_test import tfp_run_restart_1d
from modulars import exponential, print_model_info
transform_fn = exp_lognormal_moments

In [2]:
config_file = '../exp_config.json'
config = load_config(config_file)

lambda_like = config['lambda_like']
alpha_prior = config['alpha_prior']
beta_prior = config['beta_prior']
n_samples = config['n_samples']
max_iters = 100_000#config['max_iters']
data = exponential(n_samples, lambda_like)
print_model_info(
    "Gamma", "Exp", "Gamma",
    [alpha_prior, beta_prior], [lambda_like],
    data=data, n_samples=n_samples
)

conditioned_log_prob = make_conditioned_lp(
  prior_dist = tfd.Gamma(alpha_prior, beta_prior),
  likelihood_dist = lambda z: tfd.Exponential(1/z),
  x = data
)

Prior: Gamma([1.5, 1.0])
Likelihood: Exp([2.0])
Data: []


In [3]:
results = []
bij = tfb.Exp()
for seed in tqdm(range(100)):
    result = tfp_run_restart_1d(
        seed, conditioned_log_prob, bij)
    results.append(result)

  0%|          | 0/100 [00:00<?, ?it/s]

2026-03-16 21:21:48.038301: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:108] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. fit_surrogate_posterior/sanitize_seed/seed
I0000 00:00:1773710508.271241 2104855 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
%matplotlib inline
from modulars import apply_traj_transform, save_rr_tracking_csv
TRACKING_CSV = "processed_tracking/rr_tfp_gam_exp_tracking.csv"

single_means, single_stds, multi_means, multi_stds = \
apply_traj_transform(results, transform_fn=transform_fn,
                     n_samples=10_000, seed=1, NOTEBOOK=True)
save_rr_tracking_csv(
    TRACKING_CSV,
    {"default": (single_means, single_stds, multi_means, multi_stds)},
)


In [ ]:
%matplotlib inline
from modulars import load_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_tfp_gam_exp_tracking.csv"

single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="default"
)
N, T = single_means.shape

# If we have "best" from config, these should be on theta-scale (0,1)
best_mu, best_std = load_best_values(
    config=config, transform=transform_fn,
    n_samples=50_000, seed=1)


plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    best_mu, best_std, r'$\sigma^2$')
plot_mean_band_rrs_1d(
    single_means, single_stds, best_mu, best_std,
    x, r'$\sigma^2$', 1)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, best_mu, best_std,
    x, r'$\sigma^2$', 100)
